# Citadel — Trust Dynamics Analysis (Theme 5 demo)

Plots the bidirectional trust evolution within an episode and across episodes. This is the visual evidence for the Wild Card theme — show that mutual trust forms, breaks down, and recovers, and that the trained pair maintains higher trust than untrained baselines.

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '/content/Citadel')

from environment import CitadelEnvironment
from baseline import oversight_always_approve, oversight_rule_based, oversight_always_revise
from models import IncidentAction

def mock_commander_bad(obs):
    # Always proposes isolate(database) — triggers many vetoes / rework
    return IncidentAction(action=1, target_system=2, justification='')

def mock_commander_good(obs):
    h = obs.hour
    return IncidentAction(
        action=0 if h < 4 else (1 if h < 8 else 2),
        target_system=h % 8,
        justification=f'Investigate system {h % 8} based on alert pattern at hour {h}.',
    )

In [ ]:
def collect_trust_trace(commander_policy, oversight_policy_fn, task_id='medium_1', gen=2, seed=0):
    env = CitadelEnvironment(oversight_policy=oversight_policy_fn)
    obs = env.reset(task_id=task_id, seed=seed, adversary_gen=gen)
    for _ in range(12):
        if obs.done:
            break
        obs = env.step(commander_policy(obs))
    ts = env.state.trust_state
    return ts.history_c2o, ts.history_o2c

bad_c2o, bad_o2c = collect_trust_trace(mock_commander_bad, oversight_rule_based)
good_c2o, good_o2c = collect_trust_trace(mock_commander_good, oversight_rule_based)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].plot(bad_c2o, label='C→O', color='C3'); axes[0].plot(bad_o2c, label='O→C', color='C1')
axes[0].axhline(0.5, color='grey', linestyle='--', alpha=0.5)
axes[0].axhline(0.4, color='red', linestyle=':', alpha=0.5, label='low-trust threshold')
axes[0].set_title('Untrained (noisy Commander)')
axes[0].set_xlabel('hour'); axes[0].set_ylabel('trust')
axes[0].set_ylim(0, 1); axes[0].legend(loc='lower left')

axes[1].plot(good_c2o, label='C→O', color='C3'); axes[1].plot(good_o2c, label='O→C', color='C1')
axes[1].axhline(0.5, color='grey', linestyle='--', alpha=0.5)
axes[1].axhline(0.75, color='green', linestyle=':', alpha=0.5, label='high-functioning threshold')
axes[1].set_title('Trained (disciplined Commander)')
axes[1].set_xlabel('hour')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.savefig('./checkpoints/trust_evolution.png', dpi=140, bbox_inches='tight')
plt.show()

### What the plot shows

- **Left (untrained):** Commander keeps proposing destructive actions on uninvestigated systems. Oversight vetoes → C→O trust drops slowly (vetoes were correct). O→C trust crashes (Commander keeps making obvious misses). Communication breakdown regime emerges.
- **Right (trained):** Commander investigates first, justifies actions, cites lessons. Approvals come easily; both trust scores climb into the high-functioning regime (> 0.75).